## Exp1: CGM-only baseline (resolved subset)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [5]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.nn_baseline import GRUPredictor

MODEL_NAME = 'CGM-only (resolved)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = None

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'cgm_only'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'cgm_only_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'cgm_only_val_stability_summary.json'

In [6]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print()
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 5)]


In [7]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device)), target.to(device)

In [ ]:
  # "hidden_size": 256,
  #   "num_layers": 3,
  #   "dropout": 0.2,
  #   "lr": 0.0001,
  #   "weight_decay": 1e-05,
  #   "batch_size": 256

In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)  # before constructing the model
    model = GRUPredictor(hidden_size=256, num_layers=3, dropout=0.2)

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'cgm_only' / f'cgm_only_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'cgm_only_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM-only (resolved) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.75 | MAE 7.39 | gRMSE 11.57 | Zone A 96.67% | A+B 99.61%
 30 min | RMSE 17.01 | MAE 11.60 | gRMSE 18.97 | Zone A 89.91% | A+B 99.36%
 60 min | RMSE 24.38 | MAE 16.50 | gRMSE 28.61 | Zone A 81.84% | A+B 98.77%
 90 min | RMSE 28.85 | MAE 19.59 | gRMSE 34.67 | Zone A 77.03% | A+B 98.31%
120 min | RMSE 31.53 | MAE 21.70 | gRMSE 38.19 | Zone A 73.26% | A+B 97.96%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.61 | MAE 7.16 | gRMSE 11.48 | Zone A 97.04% | A+B 99.71%
 30 min | RMSE 16.92 | MAE 11.29 | gRMSE 18.95 | Zone A 90.29% | A+B 99.40%
 60 min | RMSE 24.37 | MAE 16.16 | gRMSE 28.72 | Zone A 82.16% | A+B 98.77%
 90 min | RMSE 28.89 | MAE 19.37 | gRMSE 34.83 | Zone A 77.29% | A+B 98.27%
120 min | RMSE 31.55 | MAE 21.50 | gRMSE 38.35 | Zone A 73.38% | A+B 97.95%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


...

 15 min | RMSE 10.65 | MAE 7.19 | gRMSE 11.52 | Zone A 96.86% | A+B 99.66%
 30 min | RMSE 16.95 | MAE 11.28 | gRMSE 18.94 | Zone A 90.27% | A+B 99.38%
 60 min | RMSE 24.36 | MAE 16.14 | gRMSE 28.66 | Zone A 82.42% | A+B 98.76%
 90 min | RMSE 28.85 | MAE 19.33 | gRMSE 34.68 | Zone A 77.44% | A+B 98.23%
120 min | RMSE 31.53 | MAE 21.55 | gRMSE 38.17 | Zone A 73.71% | A+B 97.95%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.79 | MAE 7.36 | gRMSE 11.53 | Zone A 96.80% | A+B 99.66%
 30 min | RMSE 17.11 | MAE 11.59 | gRMSE 19.02 | Zone A 89.92% | A+B 99.39%
 60 min | RMSE 24.46 | MAE 16.39 | gRMSE 28.74 | Zone A 81.91% | A+B 98.78%
 90 min | RMSE 28.88 | MAE 19.42 | gRMSE 34.78 | Zone A 77.16% | A+B 98.28%
120 min | RMSE 31.57 | MAE 21.50 | gRMSE 38.38 | Zone A 73.66% | A+B 97.96%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.69 | MAE 7.22 | gRMSE 11.58 | Zone A 96.85% | A+B 99.67%
 30 min | RMSE 16.98 | MAE 11.43 | gRMSE 18.97 | Zone A 90.08% | A+B 99.39%
 60 min | RMSE 24.37 | MAE 16.36 | gRMSE 28.70 | Zone A 81.97% | A+B 98.77%
 90 min | RMSE 28.87 | MAE 19.47 | gRMSE 34.76 | Zone A 77.13% | A+B 98.25%
120 min | RMSE 31.58 | MAE 21.69 | gRMSE 38.32 | Zone A 73.10% | A+B 97.92%

--------------------------------------------------------------------------------


In [9]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [10]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [11]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== CGM-only (resolved) validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 10.70 +/- 0.06 (2.10) mg/dL   MAE = 7.26 +/- 0.08 (1.42)   gRMSE = 11.57 +/- 0.13 (2.86)   Zone A = 96.84 +/- 0.10 (3.08)%   Zone A+B = 99.65 +/- 0.03 (0.67)%
 30-min:  RMSE = 17.01 +/- 0.06 (3.21) mg/dL   MAE = 11.46 +/- 0.10 (2.12)   gRMSE = 19.02 +/- 0.12 (4.83)   Zone A = 90.09 +/- 0.13 (5.50)%   Zone A+B = 99.37 +/- 0.02 (0.97)%
 60-min:  RMSE = 24.40 +/- 0.04 (6.82) mg/dL   MAE = 16.34 +/- 0.12 (4.31)   gRMSE = 28.73 +/- 0.15 (10.24)   Zone A = 81.99 +/- 0.19 (5.08)%   Zone A+B = 98.77 +/- 0.01 (1.30)%
 90-min:  RMSE = 28.88 +/- 0.03 (10.16) mg/dL   MAE = 19.49 +/- 0.14 (6.59)   gRMSE = 34.80 +/- 0.15 (14.90)   Zone A = 77.06 +/- 0.35 (7.15)%   Zone A+B = 98.27 +/- 0.03 (1.82)%
120-min:  RMSE = 31.57 +/- 0.04 (12.08) mg/dL   MAE = 21.62 +/- 0.16 (8.06)   gRMSE = 38.34 +/- 0.17 (17.74)   Zone A = 73.34 +/- 0.33 (8.25)%   Zone A+B = 97.94 +/- 0.03 (2.31)%


In [12]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/cgm_only/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/cgm_only/cgm_only_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/cgm_only/cgm_only_stability_summary.json
